# Czyszczenie danych tekstowych

Cel: usunięcie tekstów nieprzydatnych do uczenia i poprawienie artefaktów znalezionych w notebooku 01. Te same zmiany wprowadzane są we wszystkich typach tekstów.

## 1. Biblioteki

In [1]:
import pandas as pd
from pathlib import Path
from datasets import load_dataset

## 2. Wczytanie danych

In [2]:
dataset = load_dataset("DarrenJiaImbue/ai-detection-demo-dataset")

df = pd.concat([
    dataset["train"].to_pandas(),
    dataset["val"].to_pandas(),
    dataset["test"].to_pandas(),
], ignore_index=True)

df["text_raw"] = df["text"]
print("Liczba tekstów:", len(df))

Liczba tekstów: 86208


## 3. Teksty niezmienione przez model

In [3]:
unchanged = (df["text_type"] != "human_written") & (df["text"] == df["source_text"])
print("Teksty AI identyczne z oryginałem:", unchanged.sum())
print(df[unchanged]["text_type"].value_counts())

df = df[~unchanged].copy()
print("Liczba tekstów po usunięciu:", len(df))

Teksty AI identyczne z oryginałem: 14
text_type
ai_edited    14
Name: count, dtype: int64
Liczba tekstów po usunięciu: 86194


### Obserwacje

- Usunięto 14 tekstów `ai_edited` identycznych z oryginałem, czyli tyle samo, ile duplikatów znaleziono w notebooku 01. Po usunięciu zostało 86 194 tekstów.

## 4. Ujednolicenie tekstów

In [4]:
def show_changes(before, after):
    changed = before != after
    print("Zmienione teksty:", changed.sum())
    print(df.loc[changed, "text_type"].value_counts())

### 4.1. Apostrofy i cudzysłowy

In [5]:
before = df["text"]
df["text"] = (df["text"]
              .str.replace("’", "'", regex=False)
              .str.replace("‘", "'", regex=False)
              .str.replace("“", '"', regex=False)
              .str.replace("”", '"', regex=False))
show_changes(before, df["text"])

Zmienione teksty: 29580
text_type
ai_generated     12510
ai_edited        11636
human_written     5434
Name: count, dtype: int64


### Obserwacje

- Typograficzne apostrofy i cudzysłowy zamieniono w 29 580 tekstach (12 510 `ai_generated`, 11 636 `ai_edited`, 5434 `human_written`).

### 4.2. Podgląd pozostałych artefaktów

In [6]:
for pattern in [":::", "**", "{{", "Media playback"]:
    examples = df[df["text"].str.contains(pattern, regex=False)]["text"]
    print("=====", pattern, "-", len(examples), "tekstów")
    for t in examples.head(3):
        i = t.find(pattern)
        print(repr(t[max(0, i - 60): i + 100]))
    print()

===== ::: - 1178 tekstów
'Translation::: Who gets up early, finds gold.\nYou know.. time is money. If you get up early, you will have a lo'
"t. The video highlights the essential worker life been knew.::::: I bet Time's person of the year will most likely be the essential worker. That or Cuomo"
':::writing\nDay 1 (again, I think)\n\nI woke up beside the well, just like always. The bucket creaked, '

===== ** - 4556 tekstów
'o and for the mind, heart and spirit of the newly awakening!***If you suspect you may be a psychic vampire, or if you know you are...or if you think a loved one'
' Nick rekindle the faith she has set aside for other goals?"****A sweet love story with surprisingly quite a bit of suspense. I enjoyed this Heartsong Presents '
'anner treatment.We were taught that the Americans got their **** whupped after invading Canada in a nakedly hostile act of conquest. Oh I know the programme reh'

===== {{ - 3250 tekstów
" are demoralizing a better FCS opponent in EKU. thank you!

### Obserwacje

- `:::` występuje też w tekstach ludzkich jako separator, dlatego usuwane są tylko linie zawierające sam znacznik (np. `:::writing`).
- `**` w tekstach ludzkich służy też jako separator lub do cenzurowania słów, dlatego usuwane są tylko pary `**` otaczające tekst.
- `{{USERNAME}}` to zamiennik nazwy użytkownika wstawiony przez twórców zbioru tweetów.
- „Media playback is not supported on this device” stoi w osobnej linii w artykułach BBC.

### 4.3. Znaczniki :::

In [7]:
before = df["text"]
df["text"] = df["text"].str.replace(r"(?m)^:::\w*[ \t]*$", "", regex=True)
show_changes(before, df["text"])

Zmienione teksty: 1154
text_type
ai_generated    1152
ai_edited          2
Name: count, dtype: int64


### 4.4. Pogrubienia markdown

In [8]:
before = df["text"]
df["text"] = df["text"].str.replace(r"\*\*([^*\n]+)\*\*", r"\1", regex=True)
show_changes(before, df["text"])

Zmienione teksty: 4262
text_type
ai_generated     3330
ai_edited         582
human_written     350
Name: count, dtype: int64


### 4.5. Zamienniki {{...}}

In [9]:
pattern = r"\{\{[A-Z_]+\}\}"
print(df["text"].str.extract("(" + pattern + ")")[0].value_counts())
print()
print((df["text"].str.contains(pattern).groupby(df["text_type"]).mean() * 100).round(1))

0
{{USERNAME}}    3250
Name: count, dtype: int64

text_type
ai_edited        5.4
ai_generated     0.0
human_written    6.0
Name: text, dtype: float64


In [10]:
before = df["text"]
df["text"] = df["text"].str.replace(pattern, "", regex=True)
show_changes(before, df["text"])

Zmienione teksty: 3250
text_type
human_written    1715
ai_edited        1535
Name: count, dtype: int64


### 4.6. Komunikaty ze stron BBC

In [11]:
before = df["text"]
df["text"] = df["text"].str.replace(
    r"Media playback is (not supported|unsupported) on (this|your) device", "", regex=True)
show_changes(before, df["text"])
print("Pozostałe 'Media playback':", df["text"].str.contains("Media playback", regex=False).sum())

Zmienione teksty: 236
text_type
human_written    161
ai_edited         75
Name: count, dtype: int64
Pozostałe 'Media playback': 7


### 4.7. Brak spacji po kropce

In [12]:
before = df["text"]
df["text"] = df["text"].str.replace(r"([a-z][.!?])([A-Z])", r"\1 \2", regex=True)
show_changes(before, df["text"])

Zmienione teksty: 1739
text_type
human_written    1632
ai_edited         102
ai_generated        5
Name: count, dtype: int64


### 4.8. Znaki nowej linii i wielokrotne spacje

In [13]:
before = df["text"]
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()
show_changes(before, df["text"])

Zmienione teksty: 52947
text_type
ai_edited        18601
human_written    17471
ai_generated     16875
Name: count, dtype: int64


### Obserwacje

- Znaczniki `:::` usunięto z 1154 tekstów, prawie wyłącznie `ai_generated` (1152).
- Pogrubienia markdown usunięto z 4262 tekstów (3330 `ai_generated`, 582 `ai_edited`, 350 `human_written`).
- `{{USERNAME}}` występował w 6,0% tekstów ludzkich i 5,4% `ai_edited`, ale w żadnym tekście `ai_generated`. Mógłby więc zdradzać klasę. Usunięto go z 3250 tekstów.
- Komunikat „Media playback…” usunięto z 236 tekstów. W 7 tekstach został w innej formie, co przy ok. 86 tys. tekstów nie ma znaczenia.
- Brakującą spację po kropce dodano w 1739 tekstach, prawie wyłącznie ludzkich (1632).
- Znaki nowej linii i wielokrotne spacje zamieniono na pojedyncze spacje w 52 947 tekstach, we wszystkich typach w podobnej liczbie.

## 5. Teksty zbyt krótkie

In [14]:
df["n_words"] = df["text"].str.count(r"\S+")
too_short = df["n_words"] < 20
print("Teksty krótsze niż 20 słów:", too_short.sum())
print(df[too_short]["text_type"].value_counts())

df = df[~too_short].copy()
print("Liczba tekstów po usunięciu:", len(df))

Teksty krótsze niż 20 słów: 478
text_type
ai_edited        296
human_written    110
ai_generated      72
Name: count, dtype: int64
Liczba tekstów po usunięciu: 85716


### Obserwacje

- Usunięto 478 tekstów krótszych niż 20 słów (296 `ai_edited`, 110 `human_written`, 72 `ai_generated`). Po czyszczeniu zostało 85 716 tekstów.
- Teksty ludzkie miały początkowo co najmniej 20 słów, więc znalazły się poniżej progu dopiero po usunięciu `{{USERNAME}}` i komunikatów BBC.

## 6. Porównanie przed i po czyszczeniu

In [15]:
def artifact_table(text):
    checks = pd.DataFrame()
    checks["Pauza lub półpauza"] = text.str.contains(r"[—–]")
    checks["Typograficzny apostrof lub cudzysłów"] = text.str.contains(r"[’‘“”]")
    checks["Brak spacji po kropce"] = text.str.contains(r"[a-z][.!?][A-Z]")
    checks["Podwójna spacja"] = text.str.contains("  ", regex=False)
    checks["Gwiazdki (**)"] = text.str.contains("**", regex=False)
    checks["Znacznik :::"] = text.str.contains(":::", regex=False)
    checks["Znak nowej linii"] = text.str.contains("\n", regex=False)
    checks["{{USERNAME}}"] = text.str.contains("{{USERNAME}}", regex=False)
    return (checks.groupby(df["text_type"]).mean() * 100).round(1).T

In [16]:
before_table = artifact_table(df["text_raw"])
after_table = artifact_table(df["text"])
pd.concat([before_table, after_table], axis=1, keys=["przed", "po"])

przed                             \
text_type                            ai_edited ai_generated human_written   
Pauza lub półpauza                        49.3         41.3           7.7   
Typograficzny apostrof lub cudzysłów      40.8         43.6          18.9   
Brak spacji po kropce                      0.4          0.0           5.7   
Podwójna spacja                            3.3          0.4          10.9   
Gwiazdki (**)                              2.5         11.6           1.8   
Znacznik :::                               0.0          4.1           0.0   
Znak nowej linii                          61.7         58.8          43.3   
{{USERNAME}}                               5.1          0.0           5.6   

                                            po                             
text_type                            ai_edited ai_generated human_written  
Pauza lub półpauza                        49.3         41.3           7.7  
Typograficzny apostrof lub cudzysłów       0.0          0.0           0.0  
Brak spacji po kropce                      0.0          0.0           0.0  
Podwójna spacja                            0.0          0.0           0.0  
Gwiazdki (**)                              0.5          0.0           0.7  
Znacznik :::                               0.0          0.3           0.0  
Znak nowej linii                           0.0          0.0           0.0  
{{USERNAME}}                               0.0          0.0           0.0

### Obserwacje

- Po czyszczeniu typograficzne apostrofy, podwójne spacje, braki spacji po kropce, znaki nowej linii i `{{USERNAME}}` nie występują w żadnym tekście.
- Odsetek tekstów z pauzą nie zmienił się (7,7% ludzkich, 41,3% `ai_generated`), bo pauza jest cechą stylu i została zachowana.
- Zostały gwiazdki w 0,7% tekstów ludzkich (cenzurowanie słów) i `:::` w 0,3% tekstów `ai_generated` (znacznik w środku linii). Ze względu na małą skalę nie były usuwane.

## 7. Zapis wyczyszczonych danych

In [17]:
Path("../data/interim").mkdir(parents=True, exist_ok=True)

columns = ["text_id", "text", "text_raw", "text_type", "label",
           "model", "source", "source_id", "split"]
df[columns].to_parquet("../data/interim/text_cleaned.parquet", index=False)
print("Zapisano", len(df), "tekstów")

Zapisano 85716 tekstów
